# 02 LR Scheduler — CIFAR-100 with Learning Rate Scheduler

## Learning Rate Scheduler

A **learning rate scheduler** automatically adjusts the learning rate during training. Starting with a fixed learning rate can cause the model to overshoot the optimum in later epochs — a scheduler gradually reduces it for finer updates.

This notebook uses **`StepLR`**, which multiplies the learning rate by a factor `gamma` every `step_size` epochs:

```
lr = initial_lr × gamma^(epoch // step_size)
```

| Parameter | Value | Meaning |
|-----------|-------|---------|
| `step_size` | 1 | Decay the LR every 1 epoch |
| `gamma` | 0.9 | Multiply LR by 0.9 each step |

So with an initial LR of `0.001`, the schedule looks like:

| Epoch | Learning Rate |
|-------|---------------|
| 1 | 0.001000 |
| 2 | 0.000900 |
| 3 | 0.000810 |
| … | … |
| 10 | 0.000387 |

> 📌 `scheduler.step()` must be called **once per epoch** (after the inner loop), not once per mini-batch.

📄 [PyTorch LR Scheduler docs](https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

PROJECT_PATH="/content/drive/MyDrive/Practical-ML/cifar100"
!mkdir -p "{PROJECT_PATH}"
%cd "{PROJECT_PATH}"
!ls *.py

In [ ]:
# Save this cell as a Python file (Execute after editing)
%%writefile 02LRSch.py
# Adapted for CIFAR-100 with GPU + Learning Rate Scheduler
# LR Scheduler document: https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torch.optim as optim
import torch.optim.lr_scheduler as sch
import torch.backends.cudnn as cudnn

import time

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

batch_size = 32
epochs = 10

# training data preparation
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

trainset = torchvision.datasets.CIFAR100(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)
testset = torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=100, shuffle=False, num_workers=2)

# CIFAR-100 has 100 classes (20 superclasses x 5 subclasses each)

# network definition
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()

        self.conv1 = nn.Conv2d(3, 16, 3, padding=1, padding_mode='replicate')
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1, padding_mode='zeros')

        self.pool = nn.MaxPool2d(2, 2)
        self.flatten = nn.Flatten()

        self.fc1 = nn.Linear(32 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 100)  # 100 classes

    def forward(self, x):
        x = self.conv1(x) # (B,3,32,32) -> (B,16,32,32)
        x = F.relu(x)
        x = self.pool(x) # (B,16,32,32) -> (B,16,16,16)

        x = self.conv2(x) # (B,16,16,16) -> (B,32,16,16)
        x = F.relu(x)
        x = self.pool(x) # (B,32,16,16) -> (B,32,8,8)

        x = self.flatten(x) # (B,32,8,8) -> (B,32*8*8)

        x = self.fc1(x) # (B,32*8*8) -> (B,128)
        x = F.relu(x)

        x = self.fc2(x) # (B,128) -> (B,64)
        x = F.relu(x)

        x = self.fc3(x) # (B,64) -> (B,100)

        return x

net = Net()
net = net.to(device)
if device == 'cuda':
    cudnn.benchmark = True
    print('Run with GPU')
else:
    print('Run with CPU')

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)
# learning rate schduler
scheduler = sch.StepLR(optimizer, step_size=1, gamma=0.9)

net.train()
t0 = time.perf_counter()
for epoch in range(epochs):

    running_loss = 0.0
    for i, data in enumerate(trainloader):
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        if i % 100 == 99:
            t1 = time.perf_counter()
            print('[%d, %5d, %.2e] loss: %.3f, time: %.3f' %
                  (epoch + 1, i + 1, optimizer.param_groups[0]['lr'], running_loss / 2000, t1-t0))
            running_loss = 0.0
            t0 = t1
    scheduler.step() # invoke learning rate schduler after epoch

print('Finished Training')

net.eval()
correct = 0
total = 0
with torch.no_grad():
    for data in testloader:
        images, labels = data
        images, labels = images.to(device), labels.to(device)
        outputs = net(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print('Accuracy of the network on the 10000 test images: %.3f %%' % (100 * correct / total))

In [ ]:
# Execute a Python file
%cd "{PROJECT_PATH}"
!python 02LRSch.py

💾 **Don't forget to save this notebook!**

To keep your work, go to File → Save a copy in Drive before closing.

[![Return to GitHub](https://img.shields.io/badge/GitHub-Return-black?logo=github)](https://github.com/mastnk/practical-ml)